In [1]:
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

/Users/jiwon/Documents/Git-summer/ai-agent-systems/04-langgraph/education-agent/.venv/lib/python3.13/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
class LearningState(TypedDict):
    topic: str
    scenario: str
    user_answer: str
    feedback: str

In [4]:
def scenario_node(state: LearningState) -> LearningState:
    topic = state["topic"]

    prompt = f"""
    Create one short English speaking practice scenario for this topic:

    {topic}

    Make it clear and useful for an English learner.
    """

    response = llm.invoke(prompt)

    return {
        **state,
        "scenario": response.content
    }

In [6]:
def feedback_node(state: LearningState) -> LearningState:
    scenario = state["scenario"]
    user_answer = state["user_answer"]

    prompt = f"""
    You are an English speaking coach.

    Scenario:
    {scenario}

    Student's answer:
    {user_answer}

    Give feedback in 3 parts:

    1. What is good
    2. What can be improved
    3. A better version
    """

    response = llm.invoke(prompt)

    return {
        **state,
        "feedback": response.content
    }

In [7]:
graph_builder = StateGraph(LearningState)

graph_builder.add_node("scenario_node", scenario_node)
graph_builder.add_node("feedback_node", feedback_node)

graph_builder.add_edge(START, "scenario_node")
graph_builder.add_edge("scenario_node", "feedback_node")
graph_builder.add_edge("feedback_node", END)

graph = graph_builder.compile()

In [8]:
result = graph.invoke({
    "topic": "asking a professor for an assignment extension",
    "scenario": "",
    "user_answer": "Professor, I need more time. Please extend the deadline.",
    "feedback": ""
})

print("=== Scenario ===")
print(result["scenario"])

print("\n=== Feedback ===")
print(result["feedback"])

=== Scenario ===
**Scenario: Asking a Professor for an Assignment Extension**

**Setting**: You are in your professor’s office or participating in an online meeting.

**Characters**: 
- You (the student)
- Professor Smith (the professor)

---

**You**: *Knock on the door* "Hello, Professor Smith. Do you have a moment to talk?"

**Professor Smith**: "Yes, of course! How can I help you?"

**You**: "I wanted to discuss the assignment due next week for your class. I’ve been having some difficulties."

**Professor Smith**: "I see. What kind of difficulties are you facing?"

**You**: "I have been sick and unable to complete the research I need. I was wondering if it would be possible to get an extension on the assignment?"

**Professor Smith**: "How much extra time do you think you need?"

**You**: "I believe an extra week would be very helpful. I want to make sure I submit my best work."

**Professor Smith**: "That sounds reasonable. I can give you an extension until next Friday. Please tak